# Chapter 2 · 전력수요 SARIMA

**연구질문:** 전력수요 예측에 기온(외생변수)을 넣으면 SARIMA 대비 외표본 오차가 줄어드는가?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**ARIMA$(p,d,q)$:** $\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t$, $\ \varepsilon_t \sim WN(0,\sigma^2)$.

**SARIMA$(p,d,q)(P,D,Q)_m$:** 계절 차분 $(1-L^m)^D$와 계절 AR/MA 다항식 $\Phi(L^m),\Theta(L^m)$을 추가한다.

**차수 선택:** $\mathrm{AIC}=2k-2\ln\hat{L}$. AIC는 **적합 내 기준**이며 외표본 성능을 보장하지 않는다. 항상 시간순 홀드아웃에서 기준선과 비교한다.

**예측 오차 지표:** $\mathrm{MAE}$, $\mathrm{RMSE}$, $\mathrm{MASE}=\dfrac{\mathrm{MAE}}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t-y_{t-1}|}$ (분모는 학습구간 1단계 나이브의 평균절대오차; 1보다 작으면 학습구간 나이브보다 낫다).

**잔차 진단:** 잔차가 백색잡음이면 Ljung–Box $Q$ 검정에서 기각되지 않아야 한다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_openmeteo_daily(obj):
    d = obj["daily"]; idx = pd.to_datetime(d["time"])
    tmax = pd.Series(d["temperature_2m_max"], index=idx, dtype=float)
    tmin = pd.Series(d["temperature_2m_min"], index=idx, dtype=float)
    return ((tmax + tmin) / 2).rename("temp_mid")          # 최고·최저 평균(정의된 일평균 아님)

def parse_openmeteo_pm25(obj):
    h = obj["hourly"]; s = pd.Series(h["pm2_5"], index=pd.to_datetime(h["time"]), dtype=float)
    return s.resample("D").mean().rename("pm2_5")

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw)); df.columns = [c.strip().lower() for c in df.columns]
    s = pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_mid")
    if len(s) < 3000 or not s.dropna().between(-10, 40).all():
        raise ValueError("일 최저기온 길이/범위 이상")
    return s

def synthetic_temp(n=3652, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(12 + 13 * np.sin(2 * np.pi * (t - 110) / 365.25) + rng.normal(0, 3, n) + 0.0004 * t,
                     index=pd.date_range("2015-01-01", periods=n, freq="D"), name="temp_mid")

def load_energy(which="temp"):
    temp, pm, kind = None, None, "SYNTHETIC (NOT real data)"
    if USE_LIVE_DATA:
        try:
            u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
                 "&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
            temp = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=120)))
            kind = "REAL: Open-Meteo/ERA5 재분석 서울 일 기온 (관측소 실측 아님)"
        except Exception as e:
            print("Open-Meteo 기온 수집 실패 -> 2순위 시도:", type(e).__name__, str(e)[:120])
            try:
                temp = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv", timeout=60))
                kind = "REAL: Melbourne 일 최저기온 1981-1990 (jbrownlee/Datasets; 서울 아님, 일평균 아닌 최저기온)"
            except Exception as e2:
                print("일 최저기온 수집 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:100]); temp = None
        try:
            u = ("https://air-quality-api.open-meteo.com/v1/air-quality?latitude=37.57&longitude=126.98"
                 "&hourly=pm2_5&start_date=2022-09-01&end_date=2024-12-31&timezone=Asia%2FSeoul")
            pm = parse_openmeteo_pm25(json.loads(fetch_bytes(u, timeout=120)))
        except Exception as e:
            print("Open-Meteo 대기질 수집 실패(선택 항목):", type(e).__name__, str(e)[:100])
    if temp is None:
        temp = synthetic_temp()
    temp = temp.asfreq("D").interpolate()
    if which == "demand":
        rng = np.random.default_rng(SEED + 5); t = np.arange(len(temp))
        dow = temp.index.dayofweek.to_numpy()
        dem = 60 + 0.9 * np.abs(temp.to_numpy() - 18) ** 1.15 + 6 * (dow < 5) + 0.004 * t + rng.normal(0, 2.0, len(temp))
        return dict(y=pd.Series(dem, index=temp.index, name="demand"), X=temp.to_frame("temp"), m=7, freq="D",
                    label="일별 전력수요(합성, 기온 의존)", unit="GWh",
                    kind="SYNTHETIC-DERIVED (NOT real data): 수요는 합성, 기온 입력은 " + kind)
    return dict(y=temp, pm=pm, m=365, freq="D", label="일평균 기온", unit="°C", kind=kind)

D = load_energy("demand")
y, DATA_KIND, FREQ = D["y"], D["kind"], D["freq"]
M = D["m"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 분할과 기준선

In [ ]:
H = 28                                  # 예측 구간 길이(관측 단위)
LOG = False                              # 건수·양수 자료의 분산 안정화를 위한 log1p 사용 여부
s_all = y.astype(float)
z_all = np.log1p(s_all) if LOG else s_all   # 모형이 보는 척도
train_z, test_z = z_all.iloc[:-H].to_numpy(), z_all.iloc[-H:].to_numpy()
train, test = s_all.iloc[:-H].to_numpy(), s_all.iloc[-H:].to_numpy()
back = (lambda v: np.expm1(v)) if LOG else (lambda v: v)    # 모형 척도 -> 원척도
print(f"학습 {len(train)} | 검증 {len(test)} | log1p 사용: {LOG}")

In [ ]:
results = {}
def record(name, fc):
    fc = np.asarray(fc, float)
    results[name] = {"MAE": mae(test, fc), "RMSE": rmse(test, fc), "MASE": mase(test, fc, train, m=1)}
record("naive", naive_fc(train, H))
record("drift", drift_fc(train, H))
if M > 1 and M <= len(train) // 3:
    record(f"seasonal naive(m={M})", snaive_fc(train, H, M))
show(pd.DataFrame(results).T)

## 3. ARIMA / SARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

d = 0 if adfuller(train_z, autolag="AIC")[1] < 0.05 else 1       # 단순 규칙(검정은 보조 수단)
cands = []
for p in range(3):
    for q in range(3):
        try:
            fit = ARIMA(train_z, order=(p, d, q)).fit()
            cands.append((fit.aic, p, q, fit))
        except Exception:
            continue
cands.sort(key=lambda t: t[0])
aic, p_, q_, best = cands[0]
print(f"선택 ARIMA({p_},{d},{q_})  AIC={aic:.1f}  (후보 {len(cands)}개 중 AIC 최소)")
fc = best.get_forecast(H)
mean_z, ci_z = np.asarray(fc.predicted_mean), np.asarray(fc.conf_int(alpha=0.05))
record(f"ARIMA({p_},{d},{q_})", back(mean_z))
lb = acorr_ljungbox(best.resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
print("잔차 Ljung-Box(10) p =", round(float(lb), 4), "-> 작으면 잔차에 자기상관이 남음")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
USE_SARIMA = M in (7, 12) and len(train_z) > 4 * M
if USE_SARIMA:
    try:
        sfit = SARIMAX(train_z, order=(1, d, 1), seasonal_order=(1, 1, 1, M),
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        sfc = sfit.get_forecast(H)
        record(f"SARIMA(1,{d},1)(1,1,1)_{M}", back(np.asarray(sfc.predicted_mean)))
        s_mean_z, s_ci_z = np.asarray(sfc.predicted_mean), np.asarray(sfc.conf_int(alpha=0.05))
        print("SARIMA 잔차 Ljung-Box p =", round(float(acorr_ljungbox(sfit.resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0]), 4))
    except Exception as e:
        USE_SARIMA = False; print("SARIMA 적합 실패:", type(e).__name__, str(e)[:100])
else:
    print(f"M={M}: 이 노트북에서는 SARIMA를 건너뜁니다(주기가 7·12가 아니거나 표본이 짧음).")

In [ ]:
# 외생변수(기온) 포함 SARIMAX. 검증구간의 기온을 미래 시나리오 입력으로 쓴다(= 미래 기온을 안다고 가정, 결과 해석 시 명시).
X_all = D["X"]["temp"].to_numpy(float).reshape(-1, 1)
X_tr, X_te = X_all[:-H], X_all[-H:]
try:
    xfit = SARIMAX(train_z, exog=X_tr, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    xfc = np.asarray(xfit.get_forecast(H, exog=X_te).predicted_mean)
    record("SARIMAX + 기온(기온을 안다고 가정)", back(xfc))
except Exception as e:
    print("SARIMAX 외생변수 적합 실패:", type(e).__name__, str(e)[:100])

## 4. 비교와 진단

In [ ]:
tab = pd.DataFrame(results).T.sort_values("MASE")
show(tab)
best_name = tab.index[0]
print("검증구간 MASE 최소:", best_name, "| 주의: 단일 검증구간이므로 우열은 불확실(→ Ch.7 walk-forward)")
fig, ax = plt.subplots(figsize=(11, 4))
idx_te = s_all.index[-H:]
ax.plot(s_all.index[-4 * H:], s_all.iloc[-4 * H:], label="actual")
ax.plot(idx_te, back(mean_z), label=f"ARIMA({p_},{d},{q_})")
ax.fill_between(idx_te, back(ci_z[:, 0]), back(ci_z[:, 1]), alpha=0.15)
if "USE_SARIMA" in globals() and USE_SARIMA:
    ax.plot(idx_te, back(s_mean_z), label="SARIMA")
ax.plot(idx_te, snaive_fc(train, H, M) if (M > 1 and M <= len(train) // 3) else naive_fc(train, H), "--", label="baseline")
ax.legend(); ax.set_title(f"{D['label']} 예측 | {DATA_KIND}"); plt.tight_layout(); plt.show()

## 3. 결과 해석 작성
1. 어떤 기준선이 가장 강했는지, 복잡한 모형이 그 기준선을 외표본에서 이겼는지(MASE<기준선) 적는다.
2. 잔차 Ljung–Box 결과가 모형 충분성에 대해 말하는 바를 적는다.
3. AIC 최소 모형과 외표본 최소 오차 모형이 다르면 그 이유(과적합, 구조변화)를 논의한다.
4. 한계: 단일 검증구간, 로그변환 역변환 편의, 구조변화. 합성자료 결과는 방법 시연이다.

**제출:** 기준선 포함 지표표, 예측 그래프, 잔차 진단, 해석 단락, 데이터 출처·한계.